# Judge Alignment with MemAlign

This notebook aligns our evaluation judge with SME (Subject Matter Expert) feedback using the MemAlign optimizer.

**Prerequisites:**
- Run `00_setup.ipynb` to generate configuration
- Run `04-Evaluation.ipynb` to produce evaluation traces tagged with `eval: complete`
- Complete labeling sessions in the Review App (SME feedback)

**What this notebook does:**
1. Loads evaluation traces with SME feedback
2. Creates a MemAlignOptimizer to distill guidelines from SME annotations
3. Aligns the base judge to produce an aligned judge that reflects organizational preferences
4. Inspects the distilled semantic and episodic memories
5. Registers the aligned judge for use in prompt optimization

In [0]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph sentence-transformers
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import dspy
import numpy as np
from sentence_transformers import SentenceTransformer

LOCAL_EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
LOCAL_EMBEDDING_DIM = 384

class LocalSentenceTransformerEmbedder:
    def __init__(self, model=None, dimensions=None, drop_params=True, batch_size=16, **kwargs):
        self.model_name = LOCAL_EMBEDDING_MODEL
        self.dimensions = LOCAL_EMBEDDING_DIM
        self.batch_size = batch_size
        self._model = SentenceTransformer(self.model_name)

    def __call__(self, inputs, **kwargs):
        single = isinstance(inputs, str)
        texts = [inputs] if single else list(inputs)

        # BGE models work best with normalized embeddings for cosine/dot similarity.
        embeddings = self._model.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        vectors = embeddings.tolist()
        return vectors[0] if single else vectors

# MemAlign constructs dspy.Embedder internally, so patch DSPy before creating the optimizer.
dspy.Embedder = LocalSentenceTransformerEmbedder

EMBEDDING_MODEL = f"local:/{LOCAL_EMBEDDING_MODEL}"
EMBEDDING_DIM = LOCAL_EMBEDDING_DIM

print(f"Using local embeddings: {EMBEDDING_MODEL}, dim={EMBEDDING_DIM}")

Using local embeddings: local:/BAAI/bge-small-en-v1.5, dim=384


In [0]:
import json
from pathlib import Path
import mlflow
from mlflow.genai.datasets import get_dataset

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Extract configuration variables
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
DATASET_NAME = CONFIG["evaluation"]["dataset_name"]
JUDGE_MODEL = CONFIG["llm"]["judge_model"]
#ALIGNMENT_MODEL = CONFIG.get("alignment", {}).get(
#    "reflection_model",
#    CONFIG["prompt_registry"]["reflection_model"],
#)
ALIGNMENT_MODEL = "databricks:/gpt-5-4-external"
#EMBEDDING_MODEL = "databricks:/databricks-bge-large-en" #CONFIG["judges"].get("embedding_model") or "databricks:/databricks-bge-large-en"
ALIGNED_JUDGE_NAME = CONFIG['judges']['aligned_judge_name']

# Set the MLflow experiment
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

# Use the curated alignment set. These are the evaluated traces tagged after review.
traces_for_alignment = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tags.align = 'use'",
    return_type="list"
)
if not traces_for_alignment:
    traces_for_alignment = mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        filter_string="tag.align = 'use'",
        return_type="list"
    )
print(f'Found {len(traces_for_alignment)} align=use traces from evaluation dataset: {DATASET_NAME}')
print(f'Alignment reflection model: {ALIGNMENT_MODEL}')


Found 19 align=use traces from evaluation dataset: main.at_bat_assistant.atbat_assistant_eval_trace_data
Alignment reflection model: databricks:/gpt-5-4-external


In [0]:
import logging

# Set MLflow's logger to only show Errors, not Warnings
logging.getLogger("mlflow").setLevel(logging.ERROR)

In [0]:
import json
import re
from jinja2 import Template

import mlflow.genai.judges.optimizers.memalign.utils as memalign_utils
import mlflow.genai.judges.optimizers.memalign.optimizer as memalign_optimizer
from mlflow.genai.judges.optimizers.dspy_utils import construct_dspy_lm
from mlflow.genai.judges.optimizers.memalign.prompts import DISTILLATION_PROMPT_TEMPLATE


def _response_to_text(response):
    if isinstance(response, str):
        return response
    if isinstance(response, list) and response:
        return _response_to_text(response[0])
    if hasattr(response, "content"):
        return response.content
    return str(response)


def _extract_json_object(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return text[start : end + 1]
    return text


def distill_guidelines_json_mode(
    examples,
    judge_instructions: str,
    reflection_lm: str,
    existing_guidelines: list[str],
):
    if not examples:
        return []

    examples_data = [
        memalign_utils._make_json_serializable(dict(example))
        for example in examples
    ]

    indices = list(range(len(examples_data)))
    index_to_trace_id = {
        i: example._trace_id if hasattr(example, "_trace_id") else f"example_{i}"
        for i, example in enumerate(examples)
    }

    prompt = Template(DISTILLATION_PROMPT_TEMPLATE).render(
        judge_instructions=judge_instructions,
        feedback_records=examples_data,
        ids=indices,
        existing_guidelines=existing_guidelines,
        zip=zip,
        len=len,
    )

    prompt += """

Return ONLY valid JSON in this exact shape:
{
  "guidelines": [
    {
      "guideline_text": "A concrete judge guideline distilled from the feedback.",
      "source_trace_ids": [0, 1]
    }
  ]
}

Rules:
- source_trace_ids must use the numeric example ids provided above.
- Do not include markdown.
- Do not include fields other than guidelines, guideline_text, source_trace_ids.
"""

    lm = construct_dspy_lm(reflection_lm)

    try:
        raw = lm(
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )[0]
    except Exception:
        raw = lm(messages=[{"role": "user", "content": prompt}])[0]

    response_text = _extract_json_object(_response_to_text(raw))

    return memalign_utils._parse_batch_response(
        response=response_text,
        index_to_trace_id=index_to_trace_id,
        existing_guideline_texts=set(existing_guidelines),
    )


# Patch both references. optimizer.py imports distill_guidelines directly,
# so patching utils alone is not enough.
memalign_utils.distill_guidelines = distill_guidelines_json_mode
memalign_optimizer.distill_guidelines = distill_guidelines_json_mode

print("Patched MemAlign guideline distillation to JSON mode.")

Patched MemAlign guideline distillation to JSON mode.


## Create MemAlign Optimizer

The MemAlignOptimizer distills guidelines from human feedback (semantic memory) and stores representative examples (episodic memory) to calibrate the judge.

In [0]:
import mlflow
from mlflow.genai.judges import make_judge
from mlflow.genai.judges.optimizers import MemAlignOptimizer

# Create the MemAlign optimizer
optimizer = MemAlignOptimizer(
    reflection_lm=ALIGNMENT_MODEL,  # Model for distilling guidelines
    retrieval_k=3,  # Number of similar examples to retrieve
    embedding_model=EMBEDDING_MODEL,  # Model for episodic memory embeddings (from config)
)

print(f"Created MemAlignOptimizer")
print(f"  reflection_lm: {optimizer._reflection_lm}")
print(f"  retrieval_k: {optimizer._retrieval_k}")
print(f"  embedding_model: {optimizer._embedding_model}")

Created MemAlignOptimizer
  reflection_lm: databricks:/gpt-5-4-external
  retrieval_k: 3
  embedding_model: local:/BAAI/bge-small-en-v1.5


## Load Base Judge and Run Alignment

In [0]:
from mlflow.genai.scorers import get_scorer

# Load the base judge registered during evaluation
base_judge = get_scorer(name=ALIGNED_JUDGE_NAME)
print(f"Loaded base judge: {base_judge.name}")

Loaded base judge: baseball_analysis_base


In [0]:
# Align the judge using traces with SME feedback
aligned_judge = base_judge.align(traces=traces_for_alignment, optimizer=optimizer)
print(f"Alignment complete for judge: {aligned_judge.name}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Alignment complete for judge: baseball_analysis_base


## Inspect Alignment Memories

Examine the distilled guidelines (semantic memory) and stored examples (episodic memory) produced by MemAlign.

In [0]:
# Inspect semantic memory (distilled guidelines)
print("=" * 60)
print("SEMANTIC MEMORY (Distilled Guidelines)")
print("=" * 60)
for i, guideline in enumerate(aligned_judge._semantic_memory, 1):
    print(f"\n{i}. {guideline.guideline_text}")
    if guideline.source_trace_ids:
        print(f"   Source traces: {guideline.source_trace_ids[:3]}..." if len(guideline.source_trace_ids) > 3 else f"   Source traces: {guideline.source_trace_ids}")

SEMANTIC MEMORY (Distilled Guidelines)

1. When answering pitch-type percentage questions, explicitly define or enumerate what counts as the requested category (e.g., which pitch types are included as 'breaking balls') rather than giving only a number.
   Source traces: ['tr-547c546681bfb44df87d24026c94df26']

2. Assume Statcast/Genie can provide standard pitch-level fields such as pitch distribution, release speed, spin rate, and situational splits; responses that simply claim the data is unavailable for these are judged very poorly.
   Source traces: ['tr-00664e2babc0d4278f6ccb0cc18fdec8', 'tr-8f61aa5b0e9471ef827dcd5e535ba990', 'tr-65892db2c8f53c966f00b9837548e5bf']...

3. Never identify players only by internal IDs; always resolve and present human-readable player names.
   Source traces: ['tr-84f373e01b36c7ec34e1fed5e29c83dd']

4. If tools partially fail or time out, still use whatever evidence is available to provide a best-effort baseball takeaway instead of returning 'no data' w

In [0]:
# Inspect episodic memory (stored examples)
print("=" * 60)
print("EPISODIC MEMORY (Stored Examples)")
print("=" * 60)
print(f"Total examples: {len(aligned_judge._episodic_memory)}")
print(aligned_judge._episodic_memory)

EPISODIC MEMORY (Stored Examples)
Total examples: 38
[Example({'result': '2.0', 'rationale': 'I want to see what they define as breaking balls', 'inputs': '{\'request\': {\'tool_choice\': None, \'truncation\': None, \'max_output_tokens\': None, \'metadata\': None, \'parallel_tool_calls\': None, \'tools\': None, \'reasoning\': None, \'store\': None, \'stream\': None, \'temperature\': None, \'text\': None, \'top_p\': None, \'user\': None, \'background\': None, \'input\': [{\'status\': None, \'content\': "What percentage of Blake Snell\'s 2025 pitches were breaking balls?", \'role\': \'user\', \'type\': \'message\'}], \'custom_inputs\': None, \'context\': None}}', 'outputs': 'Blake\u202fSnell’s pitches in 2025 were breaking balls about **11.57\u202f%** of the time (11.565420560747663\u202f%).'}) (input_keys={'outputs', 'inputs'}), Example({'result': '1.0', 'rationale': 'statcast pitches should have the spin rate and release speed via Genie', 'inputs': "{'request': {'tool_choice': None, 't

In [0]:
# Show the aligned instructions (includes distilled guidelines)
print("=" * 60)
print("ALIGNED JUDGE INSTRUCTIONS")
print("=" * 60)
print(aligned_judge.instructions)

ALIGNED JUDGE INSTRUCTIONS
Evaluate if the response in {{ outputs }} appropriately analyzes the available data and provides an actionable recommendation to the question in {{ inputs }}. The response should be accurate, contextually relevant, and give a strategic advantage to the hitter or coaching staff making the request. Your grading criteria should be:  1: Completely unacceptable. Incorrect data interpretation or no recommendations 2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations provided with minimal strategic advantage 3: Somewhat acceptable. Relevant feedback provided with some strategic advantage 4: Mostly acceptable. Relevant feedback provided with strong strategic advantage 5: Completely acceptable. Relevant feedback provided with excellent strategic advantage

Distilled Guidelines (14):
  - When answering pitch-type percentage questions, explicitly define or enumerate what counts as the requested category (e.g., which pitch types are included as

## Update Aligned Judge

In [0]:
from mlflow.genai.scorers import ScorerSamplingConfig

mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

# Update the aligned judge in the experiment (MemAlign updates in place, no separate registration needed)
try:
    registered_aligned_judge = aligned_judge.update(
        experiment_id=EXPERIMENT_ID,
        sampling_config=ScorerSamplingConfig(sample_rate=0.0)
    )
    print(f"Updated aligned judge: {ALIGNED_JUDGE_NAME}")
except Exception as e:
    print(f"Warning updating aligned judge: {e}")

## (Optional) Re-run Evaluation with Aligned Judge

You can re-run the evaluation using the aligned judge to compare scores with the base judge.

In [0]:
# Verify we can load the aligned judge back from the experiment
mem_load_judge = get_scorer(name=ALIGNED_JUDGE_NAME)

# Inspect semantic memory (distilled guidelines)
print("=" * 60)
print("RELOADED SEMANTIC MEMORY (Distilled Guidelines)")
print("=" * 60)
for i, guideline in enumerate(mem_load_judge._semantic_memory, 1):
    print(f"\n{i}. {guideline.guideline_text}")
    if guideline.source_trace_ids:
        print(f"   Source traces: {guideline.source_trace_ids[:3]}..." if len(guideline.source_trace_ids) > 3 else f"   Source traces: {guideline.source_trace_ids}")

## Next Steps

The aligned judge is now ready for use in:
- **06-PromptOptimization.ipynb** - Use as the scorer for GEPA prompt optimization
- **07-AgentSkillsGeneration.ipynb** - Guides skill quality evaluation
- **09-Evaluation.ipynb** - Scores the held-out evaluation of both agents